#### Import Libraries

In [3]:
import csv
from collections import Counter
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules


#### Locate and Load the Groceries Dataset

In [4]:
data_file = Path("groceries.csv")

if not data_file.exists():
    raise FileNotFoundError(
        "groceries.csv was not found"
    )

transactions = []

with open(data_file, newline="", encoding="utf-8") as f:
    reader = csv.reader(f)

    for row in reader:
        items = [item.strip() for item in row if item.strip() != ""]

        if items:
            transactions.append(items)
print("Number of transactions:", len(transactions))


Number of transactions: 9835


In [5]:
n_transactions = len(transactions)

# Find all unique items
all_items = sorted(set(item for transaction in transactions for item in transaction))
n_items = len(all_items)

print("Number of transactions:", n_transactions)
print("Number of unique items:", n_items)


Number of transactions: 9835
Number of unique items: 169


#### The Top 10 Most Frequently Purchased Items

In [ ]:
item_counts = Counter(
    item
    for transaction in transactions
    for item in transaction
)

top10 = item_counts.most_common(10)

print("Top 10 items:\n")

for item, count in top10:
    support = count / n_transactions
    print(f"{item}: {count} ({support:.3f})")


#### Convert Transactions into One-Hot Encoded Data

Apriori requires the transaction data to be represented in a **one-hot encoded** format.
- `True` means the item is in the transaction `False` means the item does not appear

In [ ]:
te = TransactionEncoder()

te_array = te.fit(transactions).transform(transactions)

df = pd.DataFrame(
    te_array,
    columns=te.columns_
)

print("One-hot encoded dataset shape:", df.shape)

df.head()


#### Generate Frequent Itemsets with Apriori

Here we use a minimum support of **0.01 (1%)**.


In [ ]:
MIN_SUPPORT = 0.01

freq_itemsets = apriori(
    df,
    min_support=MIN_SUPPORT,
    use_colnames=True
)

# Add the number of items in each itemset
freq_itemsets["length"] = freq_itemsets["itemsets"].apply(len)

# Sort by support
freq_itemsets = freq_itemsets.sort_values(
    "support",
    ascending=False
).reset_index(drop=True)

multi_itemsets = freq_itemsets[
    freq_itemsets["length"] >= 2
].sort_values(
    "support",
    ascending=False
)

print("Total frequent itemsets:", len(freq_itemsets))
print("Frequent itemsets containing 2 or more items:", len(multi_itemsets))


#### Top 15 Multi-Item Frequent Itemsets

In [ ]:
multi_itemsets.head(15)


### Save Frequent Itemsets

In [ ]:
frequent_itemsets_file = Path("frequent_itemsets.csv")

multi_itemsets.head(20).to_csv(
    frequent_itemsets_file,
    index=False
)

print(f"Saved: {frequent_itemsets_file.resolve()}")


#### Generate Association Rules

Association rules help us identify relationships between items.

We use:
- **Support** — how frequently the complete rule occurs.
- **Confidence** — how often the consequent occurs when the antecedent occurs.
- **Lift** — how much more often the items occur together compared with what would be expected if they were independent.

Here, the minimum confidence is **0.30 (30%)**.

In [ ]:
MIN_CONFIDENCE = 0.30

rules = association_rules(
    freq_itemsets,
    metric="confidence",
    min_threshold=MIN_CONFIDENCE
)

rules = rules[
    ["antecedents", "consequents", "support", "confidence", "lift"]
]

rules = rules.sort_values(
    "lift",
    ascending=False
).reset_index(drop=True)

print("Total rules generated:", len(rules))


#### Format the Association Rules for Easy Reading

In [ ]:
def format_itemset(itemset):
    return ", ".join(sorted(itemset))

rules_display = rules.copy()

rules_display["antecedents"] = rules_display["antecedents"].apply(
    format_itemset
)

rules_display["consequents"] = rules_display["consequents"].apply(
    format_itemset
)

rules_display.head(15)


#### Top 15 Association Rules by Lift

In [ ]:
rules_display.head(15)


In [ ]:
# Highest Confidence Rules
rules_display.sort_values(
    "confidence",
    ascending=False
).head(3)


In [ ]:
# Highest Support Rules
rules_display.sort_values(
    "support",
    ascending=False
).head(3)


In [ ]:
# Highest Lift Rules
rules_display.sort_values(
    "lift",
    ascending=False
).head(3)


#### Saving Association Rules

In [ ]:
rules_file = Path("association_rules.csv")

rules_display.to_csv(
    rules_file,
    index=False
)

print(f"Saved: {rules_file.resolve()}")


#### Visualization: Top 10 Most Frequently Purchased Items

In [ ]:
# Prepare data for the chart
items = [item for item, count in top10]
counts = [count for item, count in top10]

plt.figure(figsize=(8, 5))

plt.barh(
    items[::-1],
    counts[::-1]
)

plt.xlabel("Number of Transactions")
plt.title("Top 10 Most Frequently Purchased Items")

plt.tight_layout()

plt.savefig(
    "top10_items.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()


#### Visualization: Top 10 Association Rules by Lift

In [ ]:
top_lift = rules_display.sort_values(
    "lift",
    ascending=False
).head(10).copy()

top_lift["rule"] = (
    top_lift["antecedents"]
    + " -> "
    + top_lift["consequents"]
)

plt.figure(figsize=(9, 5.5))

plt.barh(
    top_lift["rule"][::-1],
    top_lift["lift"][::-1]
)

plt.xlabel("Lift")
plt.title("Top 10 Association Rules by Lift")

plt.tight_layout()

plt.savefig(
    "top10_rules_lift.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()


The analysis involved loading and exploring the grocery transaction dataset, identifying the most frequently purchased items, and transforming the transactions into a one-hot encoded format. The Apriori algorithm was then used to generate frequent itemsets, followed by the generation of association rules using a minimum confidence of 30%. The resulting rules were evaluated based on support, confidence, and lift, with the frequent itemsets and association rules saved as CSV files for further analysis. Finally, two charts were created to visualize the most frequently purchased items and the top association rules based on lift.